# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# --- Setup: make this notebook work whether it's opened locally or via the Colab badge ---
import os, pathlib, subprocess

REPO_URL = "https://github.com/abc085455-byte/flyrank-ml-internship.git"

def find_repo_root(start="."):
    p = pathlib.Path(start).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "skills" / "README.md").exists() and (candidate / "docs").exists():
            return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    clone_dir = pathlib.Path("/content/flyrank-ml-internship")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

os.chdir(repo_root)
print("Working directory set to:", os.getcwd())


Working directory set to: /content/flyrank-ml-internship


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This reuses the honest, client-grouped model from `w05_model.ipynb` / audited in
`w06_validation_audit.ipynb` — same features, same label (`is_declining_label`), same
`GroupShuffleSplit(random_state=42)` on `client_id`. Under that honest split, **Logistic
Regression beat Random Forest on precision@50** (0.72 vs 0.64, from `w06`), so Logistic
Regression is the model this playbook ranks with — not Random Forest, which the *reference*
pipeline (`scripts/`, a naive random split) picked instead. That disagreement is itself a
receipt for why the split matters (Section 2 of `w06`).

The queue below is the model scored on the **7 held-out test clients** — pages the model never
trained on — because that's the only honest stand-in this starter slice has for "a brand-new
client's queue." Reason codes are transparent, threshold-based flags (adapted from the
Week-4 baseline's rule logic in `scripts/02_baseline_score.py`, extended with two things Week 5's
error analysis surfaced: a model-driven risk flag, and a `sparse_data` flag for the near-zero-data
false negatives found in `w05` cell 12).


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score

RANDOM_STATE = 42
pd.set_option("display.width", 160)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "has_search_volume", "has_word_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_search_volume"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

log_reg = Pipeline([("scaler", StandardScaler()),
                     ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
log_reg.fit(X.iloc[train_idx], y[train_idx])
proba = log_reg.predict_proba(X.iloc[test_idx])[:, 1]

queue = df.iloc[test_idx].copy().reset_index(drop=True)
queue["model_probability"] = proba
queue["y_true_for_reference_only"] = y[test_idx]  # kept only to report honesty metrics below, dropped from export

print(f"Held-out (unseen-client) queue: {len(queue)} rows across {queue['client_id'].nunique()} clients")
print(f"Model probability range: {proba.min():.3f} - {proba.max():.3f}")


Held-out (unseen-client) queue: 6163 rows across 7 clients
Model probability range: 0.023 - 0.954


In [3]:
SPARSE_DATA_IMPRESSIONS = 20  # below this, nearly every numeric feature is 0/near-0 (w05 error case #1)

def reason_codes(row):
    reasons = []
    if row["impressions_90d"] < SPARSE_DATA_IMPRESSIONS:
        reasons.append("sparse_data")            # not enough signal to trust the score either direction
        return reasons                            # sparse pages get exactly one code -- see the no-go list
    if row["trend_direction"] == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and 0 < row["engagement_rate"] < 30:
        reasons.append("low_engagement_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["model_probability"] >= 0.70:
        reasons.append("model_high_risk")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons

def suggested_action(reasons):
    r = set(reasons)
    if "sparse_data" in r:
        return "collect_more_data"          # do not score a near-empty page with confidence
    if "thin_visible_page" in r:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in r:
        return "refresh_and_review_ctr"
    if "low_engagement_visible_page" in r and "declining_with_demand" in r:
        return "refresh_and_review_engagement"
    if "stale_visible_page" in r or "declining_with_demand" in r or "page_one_decay_risk" in r:
        return "refresh"
    return "monitor"

def confidence_tier(row):
    # Confidence is downgraded for sparse-data rows regardless of how extreme the
    # probability looks -- addresses w05's false-negative case #1 (near-zero data,
    # low probability that reads as "confidently stable" but really means "no signal").
    if row["impressions_90d"] < SPARSE_DATA_IMPRESSIONS:
        return "low"
    distance = abs(row["model_probability"] - 0.5)
    if distance >= 0.30:
        return "high"
    if distance >= 0.15:
        return "medium"
    return "low"

queue["reason_codes"] = queue.apply(reason_codes, axis=1)
queue["suggested_action"] = queue["reason_codes"].apply(suggested_action)
queue["confidence"] = queue.apply(confidence_tier, axis=1)
queue["reason_codes_str"] = queue["reason_codes"].apply(lambda r: "|".join(r))
queue["priority_rank"] = queue["model_probability"].rank(method="first", ascending=False).astype(int)
queue = queue.sort_values("priority_rank").reset_index(drop=True)

print("Action mix:")
print(queue["suggested_action"].value_counts())
print("\nConfidence mix:")
print(queue["confidence"].value_counts())
print("\nTop 10 of the ranked queue:")
show_cols = ["priority_rank", "model_probability", "confidence", "suggested_action", "reason_codes_str",
             "impressions_90d", "avg_position", "days_since_last_update"]
print(queue[show_cols].head(10).to_string(index=False))


Action mix:
suggested_action
refresh_and_review_ctr           1827
monitor                          1555
refresh                          1426
collect_more_data                1298
refresh_and_review_engagement      57
Name: count, dtype: int64

Confidence mix:
confidence
low       4072
medium    1634
high       457
Name: count, dtype: int64

Top 10 of the ranked queue:
 priority_rank  model_probability confidence       suggested_action                                                               reason_codes_str  impressions_90d  avg_position  days_since_last_update
             1           0.953944       high                refresh                                          declining_with_demand|model_high_risk              128           4.2                      20
             2           0.947435       high                monitor                                                                model_high_risk              290           5.9                      20
             3       

**Archetype -> action mapping, in plain words:**

| Archetype (reason codes) | Action | Why a human trusts it |
|---|---|---|
| `sparse_data` | `collect_more_data` | Fewer than 20 impressions/90d means almost every feature is 0 — there isn't enough signal to say "refresh" or "leave it," so the honest move is "wait for data," not a guess dressed up as a score. |
| `thin_visible_page` | `expand_and_refresh` | Visible traffic, but under 1,200 words — the gap looks like depth, not just staleness. |
| `low_ctr_visible_page` | `refresh_and_review_ctr` | Good position, weak clicks — likely a title/snippet problem, not a ranking problem. |
| `low_engagement_visible_page` + `declining_with_demand` | `refresh_and_review_engagement` | Traffic is arriving but not sticking, and the trend is down — a body-content problem. |
| `stale_visible_page` / `declining_with_demand` / `page_one_decay_risk` | `refresh` | The general "worth a look" bucket — visible, aging or trending down, but no single sharper cause stood out. |
| none of the above | `monitor` | Nothing urgent flagged; check back next cycle. |

**The decay/refresh insight — stated at the level the evidence supports.** In this observed data,
older pages that get refreshed show a large, positive gap versus older pages that don't (the
FlyRank paper's own Finding #4, audited in `w06` Section 1). This dataset can't say refreshing
*causes* that gap, because which pages get refreshed isn't random — teams likely already pick
pages they believe in. **Decision-support framing:** treat "refresh" as a prioritized reviewer
suggestion, backed by an association this playbook did not test causally, not as a guaranteed
lift.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** a content/SEO reviewer (or their team lead) triaging which existing pages to look at
first, on a recurring cadence (e.g. weekly), for a client already in the portfolio.

**What it's for:** ranking a backlog of already-published content by *how worth a manual look*
each page is right now, with a stated reason — not deciding what to publish, not writing new
content, not scoring pages no one has reviewed yet for compliance or brand fit.

**Where it stops being valid:**
- **Scope:** trained and validated on this repo's 30K-row, 32-client anonymized starter slice.
  It has not been shown to work on clients, industries, or content types outside this sample —
  a new client's queue should be treated as low-confidence until re-validated on real outcomes.
- **The label's own limit:** `is_declining_label` captures *recent* direction (last 30 days vs.
  the 30 before), not chronic long-term decline — `w05`'s error case #3 (a page that's been
  flat-bad for months, not newly getting worse) shows this directly. A page can rank low here
  and still be worth a human's attention for other reasons.
- **Split-variance limit (from `w06`):** the honest, client-grouped precision@50 for this
  model ranged from 0.52 to 0.72 (mean 0.64) across five reasonable train/test splits of the same 32
  clients. Read the queue's ranking as directionally useful, not as a fixed, precise hit rate.
- **No causal claim:** nothing here says refreshing a page *will* fix it (see the decay/refresh
  insight above) — only that similar-looking pages that got refreshed tended to do better.
- **Time horizon:** this is a single trailing-90-day snapshot, not a live, continuously-updated
  feed — scores go stale as soon as new performance data exists (Section 4 covers when to
  refresh them).


In [4]:
# Make the scope limits checkable, not just asserted.
print("Training population: this repo's starter CSV only")
print(f"  rows: {len(df):,} | clients: {df['client_id'].nunique()} | content types: {df['content_type'].nunique()}")
print(f"  label window: last 30d vs. prior 30d impressions (trend_direction) -- NOT a multi-quarter trend")
print(f"  time span represented: a single trailing-90-day snapshot, no repeated per-client timestamps")

# precision@50 split-seed range, carried over from w06 Section 4 (recomputed here so this
# notebook's limits section has its own receipt, not just a citation)
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true, "score": scores})
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean())

seed_p50 = []
for seed in [42, 1, 7, 13, 99]:
    gss_s = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_s, te_s = next(gss_s.split(X, y, groups=groups))
    lr_s = Pipeline([("scaler", StandardScaler()),
                      ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
    lr_s.fit(X.iloc[tr_s], y[tr_s])
    p_s = lr_s.predict_proba(X.iloc[te_s])[:, 1]
    seed_p50.append(precision_at_k(y[te_s], p_s, 50))

print(f"\nprecision@50 (Logistic Regression, client-grouped) across 5 splits: "
      f"min={min(seed_p50):.2f}, max={max(seed_p50):.2f}, mean={np.mean(seed_p50):.2f}")
print("-> read the queue's ranking as directional, not a fixed precise hit rate.")


Training population: this repo's starter CSV only
  rows: 30,000 | clients: 32 | content types: 3
  label window: last 30d vs. prior 30d impressions (trend_direction) -- NOT a multi-quarter trend
  time span represented: a single trailing-90-day snapshot, no repeated per-client timestamps

precision@50 (Logistic Regression, client-grouped) across 5 splits: min=0.52, max=0.72, mean=0.64
-> read the queue's ranking as directional, not a fixed precise hit rate.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a human checks:**
1. **Read the actual page.** The model never sees title, body text, brand voice, or images —
   only numeric/categorical metrics. A "low CTR" flag could mean a bad title, or it could mean
   the page ranks for the wrong query entirely; only a human reading it can tell which.
2. **Check `confidence` and `reason_codes` together**, not the score alone. A `high`-probability
   row with `sparse_data` shouldn't exist by construction (Section 1 forces those to
   `collect_more_data`/`low`), but any row a reviewer finds surprising is worth a second look at
   its raw numbers before trusting the rank.
3. **Check for context the model can't see:** seasonality (a page that's "declining" every
   August on purpose), a recent site migration or redesign, a page intentionally being
   sunset, or a legal/compliance reason it exists as-is.
4. **Confirm client relevance:** this queue is scored on held-out clients as a stand-in for "a
   new client" (Section 1) — for a *real* new client, treat every row as `low` confidence until
   the model has been validated against that client's own realized outcomes.

**No-go list — must NOT be automated on this model's say-so alone:**
- **No auto-publishing or auto-editing** of any page based on the model's score or suggested
  action.
- **No auto-depublishing / auto-unpublishing / no-indexing** decisions — a false positive here
  (Section 1, `w05`'s false-positive case) could remove a page that was actually fine.
- **No automatic client-facing communication** ("your content is declining") — reason codes are
  internal triage language, not client-ready claims, and the causal-language ban in
  `writing-honest-claims` applies here too.
- **No budget or headcount decisions** driven directly by queue volume or action-mix counts.
- **No treating `sparse_data` rows as "safe" or "stable.**" They are unscored, not reassuring.


In [5]:
# Quantify what the no-go list is protecting against: how many rows would be at
# highest risk of an unreviewed wrong action, by archetype.
risk_check = queue.groupby("suggested_action").agg(
    rows=("priority_rank", "count"),
    share_low_confidence=("confidence", lambda s: round((s == "low").mean(), 3)),
).sort_values("rows", ascending=False)
print(risk_check)
print(f"\nsparse_data rows in this queue (must route to collect_more_data, never auto-act): "
      f"{(queue['reason_codes_str'] == 'sparse_data').sum()}")


                               rows  share_low_confidence
suggested_action                                         
refresh_and_review_ctr         1827                 0.596
monitor                        1555                 0.567
refresh                        1426                 0.543
collect_more_data              1298                 1.000
refresh_and_review_engagement    57                 0.509

sparse_data rows in this queue (must route to collect_more_data, never auto-act): 1298


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a single trailing-90-day snapshot with no live feedback loop yet, so these are the
signals that *would* matter once this ran on a recurring cadence:

1. **Realized-outcome drift.** Track, for pages the queue flagged `refresh`/`refresh_and_review_*`
   and a human actually acted on, whether their next 90-day trend actually improved. If the
   realized hit rate on acted-on pages falls well below the 0.52-0.72 observed range
   (Section 2), that's a retrain trigger, not just a bad week.
2. **Base-rate drift.** This snapshot's decline rate is 54.2%. If a client's or the whole
   portfolio's rate moves sharply (e.g. a big seasonal swing, or a new content type entering the
   mix), the model's `class_weight="balanced"` assumption and thresholds were tuned against a
   base rate that no longer holds.
3. **New-client growth.** The honest split only has 32 clients to draw from, which is why
   Section 2's precision@50 range is as wide as 0.52-0.72. Once meaningfully more clients exist,
   retrain and re-validate — a bigger, more diverse client pool should narrow that range, and
   it's worth checking that it actually does.
4. **Feature-distribution drift.** If `log_impressions_90d` (this playbook's Logistic Regression model's top-magnitude coefficient -- `days_with_impressions` was the top driver for Random Forest in `w05`/`w06`, but this playbook ranks with Logistic Regression, per Section 1)
   or `avg_position` shift their distribution significantly from what the model trained on —
   e.g. a GSC measurement change, a new CMS across clients — the fitted coefficients stop being
   trustworthy even if no code changed.
5. **Time-based staleness, regardless of the above.** Recompute the queue every reporting cycle
   (e.g. monthly) — a 90-day-window model scored against month-old data is already looking at a
   different reality than the one it was fit on.

**Retrain, at minimum, on any of:** realized hit rate drops meaningfully below the observed
range; the client roster grows enough to justify a bigger honest split; or a full quarter passes
without a refresh, whichever comes first.


In [6]:
# Make trigger #2 (base-rate drift) and #4 (feature drift) checkable now, as a baseline to compare future runs against.
baseline_snapshot = {
    "n_rows": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "base_rate_declining": round(float(df["is_declining_label"].mean()), 4),
    "days_with_impressions_median": round(float(df["days_with_impressions"].median()), 2),
    "avg_position_median_visible_pages": round(float(df.loc[df["avg_position"] > 0, "avg_position"].median()), 2),
    "precision_at_50_observed_range": [round(min(seed_p50), 3), round(max(seed_p50), 3)],
}
for k, v in baseline_snapshot.items():
    print(f"{k}: {v}")
print("\n-> save this snapshot (Section 5) and diff future runs against it to catch drift early.")


n_rows: 30000
n_clients: 32
base_rate_declining: 0.5421
days_with_impressions_median: 81.0
avg_position_median_visible_pages: 11.4
precision_at_50_observed_range: [0.52, 0.72]

-> save this snapshot (Section 5) and diff future runs against it to catch drift early.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Per the assignment card: the queue CSV goes to `work/outputs/` and **stays out of git on
purpose** — CI's leak-guard blocks committed dataset files, and this notebook regenerates the
CSV on every run. Figures go to `work/figures/` (committed — small SVGs, not data). Metrics and
the drift-baseline snapshot go to `work/outputs/*.json` (**committed** — they're the receipts
this playbook's numbers, and the paper's numbers next week, trace back to).


In [7]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUTS_DIR = pathlib.Path("work/outputs")
FIGURES_DIR = pathlib.Path("work/figures")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- 1) The ranked queue (CSV, gitignored by design) ---
export_cols = ["priority_rank", "content_id", "client_id", "model_probability", "confidence",
               "suggested_action", "reason_codes_str", "impressions_90d", "clicks_90d",
               "sessions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update",
               "word_count", "content_type", "main_intent"]
queue_export = queue[export_cols].rename(columns={"reason_codes_str": "reason_codes"})
queue_csv_path = OUTPUTS_DIR / "w07_ranked_action_queue.csv"
queue_export.to_csv(queue_csv_path, index=False)
print(f"Wrote {queue_csv_path} ({len(queue_export)} rows) -- gitignored, regenerate by re-running this notebook.")

# --- 2) Metrics + drift-baseline JSON (committed) ---
metrics_payload = {
    "model": "logistic_regression",
    "split_strategy": "client_grouped (GroupShuffleSplit, random_state=42)",
    "test_clients_held_out": int(queue["client_id"].nunique()),
    "precision_at_50_observed_range_across_5_seeds": [round(min(seed_p50), 3), round(max(seed_p50), 3)],
    "precision_at_50_observed_mean_across_5_seeds": round(float(np.mean(seed_p50)), 3),
    "action_mix": queue["suggested_action"].value_counts().to_dict(),
    "confidence_mix": queue["confidence"].value_counts().to_dict(),
    "sparse_data_rows": int((queue["reason_codes_str"] == "sparse_data").sum()),
    "drift_baseline_snapshot": baseline_snapshot,
}
metrics_path = OUTPUTS_DIR / "w07_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"Wrote {metrics_path} (committed -- this is the receipt).")

# --- 3) Figures (committed) ---
fig1, ax1 = plt.subplots(figsize=(6, 4))
queue["suggested_action"].value_counts().plot(kind="barh", ax=ax1, color="#3B6E8F")
ax1.set_title("Action mix -- held-out queue")
ax1.set_xlabel("pages")
fig1.tight_layout()
fig1.savefig(FIGURES_DIR / "w07_action_mix.svg")
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(6, 4))
queue["confidence"].value_counts().reindex(["high", "medium", "low"]).plot(kind="bar", ax=ax2, color="#8F5B3B")
ax2.set_title("Confidence mix -- held-out queue")
ax2.set_ylabel("pages")
fig2.tight_layout()
fig2.savefig(FIGURES_DIR / "w07_confidence_mix.svg")
plt.close(fig2)

fig3, ax3 = plt.subplots(figsize=(6, 4))
ax3.bar(range(len(seed_p50)), seed_p50, color="#4C7A4C")
ax3.axhline(np.mean(seed_p50), color="black", linestyle="--", linewidth=1, label="mean")
ax3.set_xticks(range(len(seed_p50)))
ax3.set_xticklabels(["seed 42", "seed 1", "seed 7", "seed 13", "seed 99"])
ax3.set_ylabel("precision@50")
ax3.set_title("precision@50 across 5 client-grouped splits (the range this playbook reports)")
ax3.legend()
fig3.tight_layout()
fig3.savefig(FIGURES_DIR / "w07_precision_at_50_range.svg")
plt.close(fig3)

print(f"Wrote 3 figures to {FIGURES_DIR}/ (committed):")
for p in sorted(FIGURES_DIR.glob("w07_*.svg")):
    print("  -", p)

print("\nFinal export summary:")
print(f"  queue CSV (gitignored):  {queue_csv_path}  [{len(queue_export)} rows]")
print(f"  metrics JSON (commit):   {metrics_path}")
print(f"  figures (commit):        {sorted(str(p) for p in FIGURES_DIR.glob('w07_*.svg'))}")


Wrote work/outputs/w07_ranked_action_queue.csv (6163 rows) -- gitignored, regenerate by re-running this notebook.
Wrote work/outputs/w07_playbook_metrics.json (committed -- this is the receipt).
Wrote 3 figures to work/figures/ (committed):
  - work/figures/w07_action_mix.svg
  - work/figures/w07_confidence_mix.svg
  - work/figures/w07_precision_at_50_range.svg

Final export summary:
  queue CSV (gitignored):  work/outputs/w07_ranked_action_queue.csv  [6163 rows]
  metrics JSON (commit):   work/outputs/w07_playbook_metrics.json
  figures (commit):        ['work/figures/w07_action_mix.svg', 'work/figures/w07_confidence_mix.svg', 'work/figures/w07_precision_at_50_range.svg']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.